# Stage 4 — In-Game Outcome Model and the Stakes of Each Moment

**Pipeline position:** `data/stage1_stage2_features.csv` → **Stage 4** → `data/leverage_index.parquet` → Stage 5

For every action this notebook estimates what the **next goal** would be worth, in league points, if the acting team
scores it and if it concedes it.

1. **Outcome model (§3):** a logistic regression, trained leave-one-league-out, estimates
   `P(team scores ≥1 more goal)` from goal difference, time left, time left², and home advantage.
2. **Per-frame scoring rate (§4):** each match is split into 100 frames of 54 s; the match-level probability is
   converted into a per-frame rate `θ = 1 − (1 − p)^(1/n)`.
3. **Win / draw / loss (§5):** future goals for and against are independent Binomial counts; convolving them gives
   `P(win)` and `P(draw)` at the current score and at one goal more or less.
4. **Stakes (§8b):** with expected league points `W = 3·P(win) + P(draw)`,
   `L⁺ = W(g+1) − W(g)` (gain if the team scores next) and `L⁻ = W(g) − W(g−1)` (cost if it concedes next).
   The one-sided index `LI = |ΔP(win)| + |ΔP(draw)|` for a goal *for* the team (§7–8) is kept as a baseline.

**Team strength is left out of the model on purpose.** The stakes multiply action values that already reflect the
quality of the players, so conditioning them on team strength would count it twice and penalise players on strong
teams (Kharrat, McHale & López Peña, 2020).

**Red cards** are not available as a ready-made column and are left out.

## 0. Setup and Configuration

In [ ]:
!pip install -q scikit-learn pandas numpy pyarrow scipy matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import binom

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline

print('Libraries loaded successfully.')


In [ ]:
# ── Path configuration ────────────────────────────────────────────────────────

# Repository root (the notebooks live one level below it)
BASE_DIR = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')

CSV_PATH   = BASE_DIR / 'data' / 'stage1_stage2_features.csv'

INTER_DIR  = BASE_DIR / 'intermediate'
INTER_DIR.mkdir(parents=True, exist_ok=True)

# Stage 4 outputs — metrics and plots go here.
# The primary output, leverage_index.parquet, goes to data/.
STAGE4_DIR = BASE_DIR / 'stage4_winprob'
PLOTS_DIR  = STAGE4_DIR / 'plots'
STAGE4_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
LI_PATH = BASE_DIR / 'data' / 'leverage_index.parquet'

LEAGUES = ['England', 'Spain', 'Italy', 'France', 'Germany']

# ── Binomial time-frame parameters ────────────────────────────────────────────
# Split each match into T=100 equal time-frames.
# Each frame spans 90min × 60s / 100 = 54 seconds.
# Extra-time actions (absolute_second > 5400) are clipped to frame 99,
# giving t_remaining = 1 — reflecting that very little time is left.
T_FRAMES       = 100
SECS_PER_FRAME = (90 * 60) / T_FRAMES   # 54.0 seconds per frame

# Truncate the Binomial PMF at MAX_GOALS additional goals per team.
# Captures >99.9% of probability mass at realistic football scoring rates (~0.015/frame).
MAX_GOALS = 6

RANDOM_STATE = 42

print('Configuration ready.')
print(f'  CSV path:         {CSV_PATH}')
print(f'  Stage 4 dir:      {STAGE4_DIR}')
print(f'  Seconds per frame: {SECS_PER_FRAME:.1f}s  ({T_FRAMES} frames / 90 min)')
print(f'  Output:           {LI_PATH}')

## 1. Load Data

We load only the columns required for the win-probability model and LI computation. The full CSV is ~1.55 GB; loading a column subset keeps memory usage manageable (~400–600 MB for this subset).

**Timing note:** Always use `absolute_second` (seconds since kick-off of the *match*, accounting for both halves) rather than `time_seconds` (which resets to 0 at half-time and gives incorrect match-minute values).

In [ ]:
WIN_PROB_COLS = [
    'game_id', 'league_country', 'gameweek', 'period_id',
    'absolute_second', 'minute',
    'team_id', 'player_id',
    'home_team_id', 'away_team_id', 'is_home_team',
    'home_goals_before', 'away_goals_before',
    'goals_for', 'goals_against', 'goal_difference',
    'score_margin', 'score_state',
    'is_post_75',
    'minutes_remaining',
    'home_score', 'away_score',   # final scores — used to build training targets
]

print('Loading selected columns from CSV (~2–3 min for 2.8M rows)...')
df = pd.read_csv(CSV_PATH, usecols=WIN_PROB_COLS, low_memory=False)
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

In [ ]:
# ── Assign time-frame index t (0–99) and frames remaining ────────────────────
# absolute_second runs 0 → ~5400s for a 90-min match, plus extra time.
# We bin into T=100 frames (each 54 s) and clip extra time to frame 99.
# t_remaining is clipped to a minimum of 1 to prevent division-by-zero
# during the theta inversion step (§4), even for the very last action.

df['t_frame']     = (df['absolute_second'] / SECS_PER_FRAME).clip(upper=T_FRAMES - 1).astype(int)
df['t_remaining'] = (T_FRAMES - df['t_frame']).clip(lower=1)   # always >= 1

print('Time-frame columns added.')
print(df[['absolute_second', 'minute', 't_frame', 't_remaining']].describe().round(2))


## 2. Build Training Targets

### 2.1 Match-level targets

For the logistic regression we build two binary targets per action:

- **`team_will_score`** = 1 if the acting team scores ≥1 more goal after this action.
- **`team_will_concede`** = 1 if the opposing team scores ≥1 more goal after this action.

These are derived by comparing `goals_before` (goals scored so far) against the final match score.

> **Data quality note (from dataset guide):** Goal reconstruction has a 95% agreement rate with official scores due to minor Wyscout tagging inconsistencies. A small number of rows may have `goals_remaining < 0`; we clip to 0.

### 2.2 Why these are NOT θ

`P(team_will_score)` ≈ 0.5–0.7 is the probability of scoring *at least once* in the remaining match time. The Binomial parameter θ must be the probability of scoring in *one 54-second frame* ≈ 0.01–0.02. The two are linked by:

```
P(score ≥1 in n frames) = 1 − (1 − θ)^n
```

We train logistic regression to predict `P(score ≥1)` and then invert this formula to recover the correct θ in §4.


In [ ]:
# ── Goals remaining for each team after this action ───────────────────────────
# Clip to 0 to handle the ~7.5% of rows where Wyscout event tagging
# disagrees slightly with official match records (goals_remaining can be -1).
df['home_goals_remaining'] = (df['home_score'] - df['home_goals_before']).clip(lower=0)
df['away_goals_remaining'] = (df['away_score'] - df['away_goals_before']).clip(lower=0)

# ── Binary targets: does the team / opponent score at least once more? ────────
df['home_will_score'] = (df['home_goals_remaining'] > 0).astype(int)
df['away_will_score'] = (df['away_goals_remaining'] > 0).astype(int)

df['team_will_score']   = np.where(df['is_home_team'] == 1,
                                   df['home_will_score'], df['away_will_score'])
df['team_will_concede'] = np.where(df['is_home_team'] == 1,
                                   df['away_will_score'], df['home_will_score'])

print('Training targets constructed.')
print(f"  team_will_score   mean: {df['team_will_score'].mean():.3f}  "
      f"({df['team_will_score'].mean()*100:.1f}% of actions have the team scoring again)")
print(f"  team_will_concede mean: {df['team_will_concede'].mean():.3f}")


### 2.3 Feature set for the win-probability logistic regression

| Feature | Rationale |
|---|---|
| `goal_difference` | Current score margin (acting team perspective). A trailing team is less likely to score and more likely to concede. |
| `t_remaining` | Frames remaining. More time directly increases `P(score ≥1)`. This is the primary driver of the shape of the Binomial distribution. |
| `t_remaining_sq` | Quadratic term. The relationship between remaining time and scoring probability is non-linear: gaining 10 extra frames at the end of a match matters far more than 10 frames in the first half. |
| `is_home_team` | Home teams score at a higher rate than away teams at equivalent strength. |

**Team strength is not a feature.** The stakes multiply an action value that already reflects player quality, so including it would count team strength twice and penalise players on strong teams.

In [ ]:
# ── Quadratic time feature ────────────────────────────────────────────────────
# Captures the non-linear urgency effect: the final 5 frames of a match
# are more impactful on scoring probability than 5 frames early in the game.
df['t_remaining_sq'] = df['t_remaining'] ** 2

WIN_FEATURES = [
    'goal_difference',   # signed score difference: acting team minus opponent
    't_remaining',       # linear time pressure (1–100 frames)
    't_remaining_sq',    # quadratic time pressure
    'is_home_team',      # binary home-advantage indicator
]

print(f'Feature matrix shape: {df[WIN_FEATURES].shape}')
print('\nFeature descriptive statistics:')
print(df[WIN_FEATURES].describe().round(3))

## 3. Leave-One-League-Out Training of the Win-Probability Model

We apply the same LOLO cross-validation strategy as Stage 3 to prevent data leakage. With only one season available (2017/18), training and evaluating on the same matches would let the model see its own outcomes.

**LOLO structure:** each fold trains on four leagues (~1,446 matches) and predicts on the fifth.

**Model:** `LogisticRegression(C=1.0)` inside a `StandardScaler` pipeline. StandardScaler is essential because `t_remaining` (range 1–100) and `t_remaining_sq` (range 1–10,000) would otherwise dominate `goal_difference` (range ≈ −5 to +5) in the coefficient updates.

**Two models per fold:**
- *Score model*: predicts `P(acting team scores ≥1)` from normal game-state features.
- *Concede model*: predicts `P(opponent scores ≥1)` by negating the goal difference and swapping the home-team flag to model the problem from the opponent's perspective — reusing the same logistic regression without needing separately labelled data.

**What we store:** raw logistic regression outputs (`p_will_score`, `p_will_concede`). These are the match-level "will score" probabilities — θ conversion happens in §4.

In [ ]:
LOLO_FOLDS = {
    'England': ['Spain',   'Italy', 'France', 'Germany'],
    'Spain':   ['England', 'Italy', 'France', 'Germany'],
    'Italy':   ['England', 'Spain', 'France', 'Germany'],
    'France':  ['England', 'Spain', 'Italy',  'Germany'],
    'Germany': ['England', 'Spain', 'Italy',  'France'],
}

def make_lr_pipeline():
    """Logistic regression in a StandardScaler pipeline.
    C=1.0 applies moderate L2 regularisation.
    max_iter=500 ensures convergence on this feature set.
    """
    return Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_STATE))
    ])

# ── Opponent's point of view (used by the concede model) ─────────────────────
# The goal difference changes sign and the home flag swaps;
# t_remaining and t_remaining_sq are the same for both teams.
NEGATE_COLS = [WIN_FEATURES.index('goal_difference')]
HOME_COL    = WIN_FEATURES.index('is_home_team')

def to_opponent_view(X):
    """Return a copy of X seen from the opponent's side."""
    X = X.astype(float)
    X[:, NEGATE_COLS] = -X[:, NEGATE_COLS]
    X[:, HOME_COL]    = 1 - X[:, HOME_COL]
    return X

# ── Columns to hold logistic regression outputs (filled per fold below) ───────
# p_will_score / p_will_concede: match-level P(score ≥1) — NOT per-frame theta
# theta_score / theta_concede:   per-frame rates after CDF inversion (filled in §4)
for col in ['p_will_score', 'p_will_concede',
            'theta_score', 'theta_concede',
            'p_win', 'p_draw', 'p_loss']:
    df[col] = np.nan

fold_metrics = []

for held_out, train_leagues in LOLO_FOLDS.items():
    print(f'\n── Fold: held-out = {held_out} ──')

    train_mask = df['league_country'].isin(train_leagues)
    val_mask   = df['league_country'] == held_out

    X_train = df.loc[train_mask, WIN_FEATURES].values
    X_val   = df.loc[val_mask,   WIN_FEATURES].values

    # ── Score model ───────────────────────────────────────────────────────────
    # Predicts P(acting team scores ≥1 more goal in the rest of the match).
    y_score_train = df.loc[train_mask, 'team_will_score'].values
    y_score_val   = df.loc[val_mask,   'team_will_score'].values

    score_pipe = make_lr_pipeline()
    score_pipe.fit(X_train, y_score_train)
    p_score_val = score_pipe.predict_proba(X_val)[:, 1]

    # ── Concede model ─────────────────────────────────────────────────────────
    # P(opponent scores ≥1): the same state, shown from the opponent's side.
    X_train_opp = to_opponent_view(X_train)
    X_val_opp   = to_opponent_view(X_val)

    y_concede_train = df.loc[train_mask, 'team_will_concede'].values
    y_concede_val   = df.loc[val_mask,   'team_will_concede'].values

    concede_pipe = make_lr_pipeline()
    concede_pipe.fit(X_train_opp, y_concede_train)
    p_concede_val = concede_pipe.predict_proba(X_val_opp)[:, 1]

    # ── Store raw logistic regression outputs ──────────────────────────────────
    df.loc[val_mask, 'p_will_score']   = p_score_val
    df.loc[val_mask, 'p_will_concede'] = p_concede_val

    # ── Fold evaluation metrics ────────────────────────────────────────────────
    bs_score    = brier_score_loss(y_score_val,   p_score_val)
    auc_score   = roc_auc_score(y_score_val,      p_score_val)
    bs_concede  = brier_score_loss(y_concede_val, p_concede_val)
    auc_concede = roc_auc_score(y_concede_val,    p_concede_val)

    fold_metrics.append({
        'fold':                held_out,
        'brier_score_model':   round(bs_score,    4),
        'auc_score_model':     round(auc_score,   4),
        'brier_concede_model': round(bs_concede,  4),
        'auc_concede_model':   round(auc_concede, 4),
    })
    print(f'  Score model   — Brier: {bs_score:.4f}  AUC: {auc_score:.4f}')
    print(f'  Concede model — Brier: {bs_concede:.4f}  AUC: {auc_concede:.4f}')

print('\nAll LOLO folds complete.')

In [ ]:
# ── Save fold metrics table ────────────────────────────────────────────────────
metrics_df = pd.DataFrame(fold_metrics)

# Add a MEAN row for quick reference
mean_vals           = metrics_df.drop(columns='fold').mean().to_dict()
mean_vals['fold']   = 'MEAN'
metrics_df          = pd.concat([metrics_df, pd.DataFrame([mean_vals])], ignore_index=True)

print(metrics_df.to_string(index=False))
metrics_df.to_csv(STAGE4_DIR / 'winprob_fold_metrics.csv', index=False)
print(f"\nSaved: {STAGE4_DIR / 'winprob_fold_metrics.csv'}")


## 4. Convert Match-Level Probability to Per-Frame Scoring Rate θ

The logistic regression outputs `p = P(team scores ≥1 goal in t_remaining frames)`. The Binomial model requires θ = `P(team scores in ONE frame)`. These are linked by the Binomial CDF:

```
P(score ≥ 1 in n frames) = 1 − (1 − θ)^n

Solving for θ:   θ = 1 − (1 − p)^(1/n)
```

**Numerical example at minute 85 (t_remaining ≈ 6 frames):**

| Logistic output `p` | Formula | θ (per frame) | Expected goals in 6 frames |
|---|---|---|---|
| 0.09 (trailing by 2) | 1 − 0.91^(1/6) | ≈ 0.0155 | 0.093 |
| 0.18 (level game)    | 1 − 0.82^(1/6) | ≈ 0.0328 | 0.197 |
| 0.30 (leading by 1)  | 1 − 0.70^(1/6) | ≈ 0.0582 | 0.349 |


**Edge case handling:**
- `t_remaining` is already clipped to ≥1 (§1), so `1/n` never causes division-by-zero.
- `p` is clipped to `[ε, 1−ε]` where ε = 1e-9 to ensure `(1−p)^(1/n)` is numerically stable at the boundaries.

In [ ]:
# ── Numerical safety clip ─────────────────────────────────────────────────────
# p=0 → theta=0 (team has no scoring chance — fine).
# p=1 → theta=1 (team scores in every frame — fine mathematically, but rare).
# We clip away from exact 0 and 1 to avoid floating-point edge cases in the
# subsequent power operation and Binomial convolution.
EPS = 1e-9
p_score_safe   = df['p_will_score'].clip(EPS, 1 - EPS).values
p_concede_safe = df['p_will_concede'].clip(EPS, 1 - EPS).values
n = df['t_remaining'].values   # already clipped to >= 1 in §1

# ── The inversion: θ = 1 − (1 − p)^(1/n) ────────────────────────────────────
# theta_score:   per-frame P(acting team scores one goal)
# theta_concede: per-frame P(opponent scores one goal)
# These should be in the range 0.005–0.05 — small, matching real football rates.
df['theta_score']   = 1.0 - (1.0 - p_score_safe)   ** (1.0 / n)
df['theta_concede'] = 1.0 - (1.0 - p_concede_safe) ** (1.0 / n)

# ── Sanity check: theta values must be small ───────────────────────────────────
# The average scoring rate across all match states is ~1.35 goals/team/90min
# = 1.35 / 100 frames = 0.0135 per frame.
# Theta will be higher in the clutch window (time pressure squeezes scoring rate).
print('Per-frame theta statistics — should be small (~0.01–0.05):')
for col in ['theta_score', 'theta_concede']:
    s = df[col]
    print(f'  {col:<20s}  mean={s.mean():.4f}  median={s.median():.4f}  '
          f'p5={s.quantile(.05):.4f}  p95={s.quantile(.95):.4f}  max={s.max():.4f}')

# ── Cross-check at minute 85–89, level games ──────────────────────────────────
# Expected value: theta * t_remaining ≈ 0.015 * 6 ≈ 0.09 goals per team
# over the remaining ~6 frames — plausible for a tight late-game moment.
late_level = (df['minute'].between(85, 89)) & (df['score_state'] == 'level')
print(f"\nMean theta_score at min 85–89, level game: "
      f"{df.loc[late_level, 'theta_score'].mean():.4f}  "
      f"(x6 frames ≈ {df.loc[late_level, 'theta_score'].mean() * 6:.3f} expected goals)")


## 5. Win / Draw / Loss Probabilities via Binomial Convolution

Given the per-frame scoring rates θ_score and θ_concede (now correctly small values), we model each team's additional goals in the remaining frames as independent Binomial distributions:

```
Goals scored   ~ Binomial(t_remaining, theta_score)
Goals conceded ~ Binomial(t_remaining, theta_concede)
```

**Convolution logic:**
For each pair of possible outcomes (k_scored, k_conceded), the joint probability is `P(k_scored) × P(k_conceded)`. The resulting `final_goal_diff = current_gd + k_scored − k_conceded` determines win / draw / loss. We sum over all pairs.

**Performance optimisation:**  
We loop over unique `t_remaining` values (at most 100: frames 1–100) and process all rows with the same `t_remaining` as a batch. NumPy's `binom.pmf` is vectorised over the batch dimension, so the double inner loop (over k_scored and k_conceded, at most 7×7=49 iterations) is the only Python-level bottleneck. Runtime is well under a minute for all three scorelines on 2.8M rows.

**Current state and both counterfactuals in one pass:** the PMF grids and the joint probabilities depend only on θ and `t_remaining`, never on the scoreline. `binomial_outcome_probs_shifted` therefore builds them once per batch and evaluates the outcome at `g`, `g+1` (acting team scores next) and `g−1` (acting team concedes next) together. θ is the same in all three: a goal changes the scoreline, not the teams' scoring rates or the time left. §7 checks the one-pass result against `binomial_outcome_probs` on a random sample.

**Truncation at MAX_GOALS = 6:**  
At θ ≈ 0.015 and n ≤ 100, `P(score > 6 goals) < 0.001%` — well below any meaningful precision threshold.


In [ ]:
def binomial_outcome_probs(theta_score, theta_concede, t_remaining, goal_diff,
                           max_goals=MAX_GOALS):
    """
    Compute P(win), P(draw), P(loss) from the acting team's perspective,
    using independent Binomial models for future goals scored and conceded.

    Parameters
    ----------
    theta_score   : array (N,)  Per-frame P(acting team scores one goal).
                                Must be small values ~0.01–0.05 (after CDF inversion).
    theta_concede : array (N,)  Per-frame P(opponent scores one goal).
    t_remaining   : array (N,)  Integer frames remaining in the match (1–100).
    goal_diff     : array (N,)  Current signed goal difference (acting − opponent).
                                Positive = acting team is leading.
    max_goals     : int         PMF truncation threshold (default 6).

    Returns
    -------
    p_win, p_draw, p_loss : arrays (N,), each in [0, 1], summing to ≈ 1.0.
    """
    theta_score   = np.asarray(theta_score,   dtype=float)
    theta_concede = np.asarray(theta_concede, dtype=float)
    t_remaining   = np.asarray(t_remaining,   dtype=int)
    goal_diff     = np.asarray(goal_diff,      dtype=float)

    N      = len(theta_score)
    p_win  = np.zeros(N)
    p_draw = np.zeros(N)
    p_loss = np.zeros(N)

    ks = np.arange(max_goals + 1)   # goal counts: 0, 1, 2, …, max_goals

    # Outer loop: unique t_remaining values (at most 100 iterations)
    for t_rem in np.unique(t_remaining):
        mask = t_remaining == t_rem
        if not mask.any():
            continue

        ts  = theta_score[mask]    # shape (M,)
        tc  = theta_concede[mask]  # shape (M,)
        gd  = goal_diff[mask]      # shape (M,)

        # PMF grids — shape (M, max_goals+1)
        # p_score_grid[i, k] = P(team scores exactly k goals | theta_score[i], t_rem)
        p_score_grid   = binom.pmf(ks[np.newaxis, :], t_rem, ts[:, np.newaxis])
        p_concede_grid = binom.pmf(ks[np.newaxis, :], t_rem, tc[:, np.newaxis])

        pw      = np.zeros(mask.sum())
        pd_draw = np.zeros(mask.sum())
        pl      = np.zeros(mask.sum())

        # Inner double loop: 7 × 7 = 49 iterations maximum
        # joint[i] = P(team scores k_s AND concedes k_c)
        # final_gd[i] = current goal diff + goals scored − goals conceded
        for k_s in range(max_goals + 1):
            for k_c in range(max_goals + 1):
                joint    = p_score_grid[:, k_s] * p_concede_grid[:, k_c]
                final_gd = gd + k_s - k_c
                pw      += joint * (final_gd > 0)
                pd_draw += joint * (final_gd == 0)
                pl      += joint * (final_gd < 0)

        p_win[mask]  = pw
        p_draw[mask] = pd_draw
        p_loss[mask] = pl

    return p_win, p_draw, p_loss


def binomial_outcome_probs_shifted(theta_score, theta_concede, t_remaining, goal_diff,
                                   shifts=(0, 1, -1), max_goals=MAX_GOALS):
    """
    Same model as binomial_outcome_probs, evaluated at several goal-difference
    shifts in ONE pass. The PMF grids and joint probabilities depend only on
    theta and t_remaining, so they are built once per batch and reused.

    Returns
    -------
    dict  shift -> (p_win, p_draw, p_loss), each an array (N,).
    """
    theta_score   = np.asarray(theta_score,   dtype=float)
    theta_concede = np.asarray(theta_concede, dtype=float)
    t_remaining   = np.asarray(t_remaining,   dtype=int)
    goal_diff     = np.asarray(goal_diff,     dtype=float)

    N   = len(theta_score)
    out = {s: (np.zeros(N), np.zeros(N), np.zeros(N)) for s in shifts}
    ks  = np.arange(max_goals + 1)

    for t_rem in np.unique(t_remaining):
        mask = t_remaining == t_rem
        ts, tc, gd = theta_score[mask], theta_concede[mask], goal_diff[mask]

        p_score_grid   = binom.pmf(ks[np.newaxis, :], t_rem, ts[:, np.newaxis])
        p_concede_grid = binom.pmf(ks[np.newaxis, :], t_rem, tc[:, np.newaxis])

        acc = {s: [np.zeros(mask.sum()) for _ in range(3)] for s in shifts}
        for k_s in range(max_goals + 1):
            for k_c in range(max_goals + 1):
                joint = p_score_grid[:, k_s] * p_concede_grid[:, k_c]
                for s in shifts:
                    final_gd = gd + s + k_s - k_c
                    acc[s][0] += joint * (final_gd > 0)
                    acc[s][1] += joint * (final_gd == 0)
                    acc[s][2] += joint * (final_gd < 0)

        for s in shifts:
            for j in range(3):
                out[s][j][mask] = acc[s][j]

    return out

print(f'Binomial convolution functions defined.')
print(f'  MAX_GOALS = {MAX_GOALS}  (49 inner-loop iterations per unique t_remaining)')
print(f'  Unique t_remaining values: {np.unique(df["t_remaining"].values).shape[0]}')


In [ ]:
# ── Current state and both counterfactuals, in one pass ───────────────────────
#   shift  0 → p_win,        p_draw,        p_loss   (current scoreline)
#   shift +1 → p_win_plus1,  p_draw_plus1            (acting team scores next)
#   shift −1 → p_win_minus1, p_draw_minus1           (acting team concedes next)
print('Computing win/draw/loss probabilities for all 2.8M rows at g, g+1 and g-1...')
print('(One pass per unique t_remaining value — well under a minute)')

probs = binomial_outcome_probs_shifted(
    theta_score   = df['theta_score'].values,
    theta_concede = df['theta_concede'].values,
    t_remaining   = df['t_remaining'].values,
    goal_diff     = df['goal_difference'].values,
    shifts        = (0, 1, -1),
)
df['p_win'],        df['p_draw'],        df['p_loss'] = probs[0]
df['p_win_plus1'],  df['p_draw_plus1'],  _            = probs[1]
df['p_win_minus1'], df['p_draw_minus1'], _            = probs[-1]
del probs

# ── Clip to [0, 1] ────────────────────────────────────────────────────────────
# The Binomial PMF truncated at MAX_GOALS leaves a tiny residual mass
# (probability of scoring > 6 goals) that can push the sum fractionally
# below 1.0. Values remain well within [0, 1] but clipping prevents any
# downstream sklearn function from raising a ValueError on out-of-range inputs.
PROB_COLS = ['p_win', 'p_draw', 'p_loss',
             'p_win_plus1', 'p_draw_plus1', 'p_win_minus1', 'p_draw_minus1']
for col in PROB_COLS:
    df[col] = df[col].clip(0.0, 1.0)

# ── Sanity check: probabilities should sum to ≈ 1.0 ──────────────────────────
prob_sum = df[['p_win', 'p_draw', 'p_loss']].sum(axis=1)
print(f'\nProbability sum — mean: {prob_sum.mean():.6f}  '
      f'min: {prob_sum.min():.6f}  max: {prob_sum.max():.6f}')
print('(Should be within floating-point precision of 1.000000)')
print('Done.')


## 6. Validate the Outcome Model

The calibration curve (predicted `P(win)` against the empirical win rate) should follow the diagonal.

**Expected behaviour**
- Leading teams: `P(win)` near 1 late in the match; trailing teams near 0; level games in between, with a high `P(draw)`.
- At kick-off, `P(win)` reflects home advantage; in the final frame it is near 0 or 1 depending on the score.

In [ ]:
# ── Compute empirical win outcome for the acting team ─────────────────────────
home_win          = (df['home_score'] > df['away_score']).astype(int)
away_win          = (df['away_score'] > df['home_score']).astype(int)
df['actual_win']  = np.where(df['is_home_team'] == 1, home_win, away_win)

# ── Prepare calibration inputs ─────────────────────────────────────────────────
p_win_cal  = df['p_win'].clip(0.0, 1.0)
valid_mask = p_win_cal.notna() & df['actual_win'].notna()
p_win_cal  = p_win_cal[valid_mask].values
y_true_cal = df.loc[valid_mask, 'actual_win'].values

print(f'Calibration sample: {len(p_win_cal):,} rows')
print(f'P(win) outside [0,1] after clip: {((p_win_cal < 0) | (p_win_cal > 1)).sum()}  (should be 0)')

# ── Calibration curve (15 quantile bins) ──────────────────────────────────────
prob_true, prob_pred = calibration_curve(
    y_true_cal, p_win_cal, n_bins=15, strategy='quantile'
)

state_colors = {'chasing': '#e74c3c', 'level': '#3498db', 'protecting': '#2ecc71'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: calibration plot
ax = axes[0]
ax.plot(prob_pred, prob_true, 'o-', label='Win probability model', linewidth=1.8, markersize=6)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Predicted P(win)')
ax.set_ylabel('Empirical win rate')
ax.set_title('Win Probability Calibration')
ax.legend()
ax.grid(alpha=0.3)

# Right: distribution of P(win) by score state
ax2 = axes[1]
for state, col in state_colors.items():
    sub = df[df['score_state'] == state]
    ax2.hist(sub['p_win'].clip(0.0, 1.0), bins=40, alpha=0.5,
             label=state, color=col, density=True)
ax2.set_xlabel('P(win)')
ax2.set_ylabel('Density')
ax2.set_title('Win Probability Distribution by Score State')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'win_prob_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PLOTS_DIR / 'win_prob_calibration.png'}")

In [ ]:
# ── Boundary checks ───────────────────────────────────────────────────────────
# Kick-off (t_frame == 0): should broadly reflect home advantage
kickoff = df[df['t_frame'] == 0].drop_duplicates('game_id')
print(f'Kick-off win probability (n={len(kickoff):,} unique matches):')
print(kickoff[['p_win', 'p_draw', 'p_loss']].describe().round(3))

# End-of-match (t_frame == 99): should be near 0 or 1 depending on scoreline
end_game = df[df['t_frame'] == 99]
print(f'\nEnd-game P(win) by score state (n={len(end_game):,} rows):')
for state in ['protecting', 'level', 'chasing']:
    sub = end_game[end_game['score_state'] == state]
    print(f'  {state:<12s}: mean P(win) = {sub["p_win"].mean():.3f}')
print('Expected: protecting → near 1.0, level → near 0.5, chasing → near 0.0')


## 7. Leverage Index Computation

> **Role in the pipeline.** §7–8 compute the *symmetric* Leverage Index. It is kept as an **evaluation baseline**: comparing it with the signed weights isolates what separating scoring from conceding adds. The weights passed to Stage 5 are the signed expected-points leverage `L⁺` / `L⁻` in §8b.

The Leverage Index answers the question: **"How much would the match outcome shift if the acting team scored right now?"**

**Formula:**
```
LI(x_t) = |P(win | team scores at t)  − P(win | current state)|
         + |P(draw | team scores at t) − P(draw | current state)|
```

The counterfactual "team scores at t" (`goal_difference + 1`) was computed in §5, in the same pass as the current state. The absolute values are needed because:
- Scoring can *increase* P(draw) (equaliser for a trailing team) or *decrease* it (go-ahead goal for a tied team).
- Scoring always increases P(win) for both teams in different contexts.

Using absolute values ensures LI is always non-negative regardless of the direction of each shift.

**Same θ for the counterfactual:** We keep θ unchanged when computing the hypothetical "team scores" scenario. θ represents the underlying scoring rate based on team strength and time remaining — not the current score. Scoring a goal does not change how strong the teams are or how much time is left.

**Intuitive examples:**
- Equaliser at the 88th minute of a 1-0 game → massive ΔP(draw) and ΔP(win) → high LI.
- Goal at the 70th minute of a 4-0 game → negligible shift in any probability → near-zero LI.

In [ ]:
# ── Counterfactuals were computed in §5, in the same pass as the current state:
#     g+1 → p_win_plus1,  p_draw_plus1    (acting team scores next)
#     g−1 → p_win_minus1, p_draw_minus1   (acting team concedes next)
#
# Regression check: the one-pass function must reproduce the reference
# single-scoreline convolution exactly, for every shift, on a random sample.
rng  = np.random.default_rng(RANDOM_STATE)
samp = df.iloc[rng.choice(len(df), size=min(200_000, len(df)), replace=False)]

SHIFT_COLS = {0: ('p_win', 'p_draw'),
              1: ('p_win_plus1', 'p_draw_plus1'),
             -1: ('p_win_minus1', 'p_draw_minus1')}
for shift, (col_w, col_d) in SHIFT_COLS.items():
    ref_w, ref_d, _ = binomial_outcome_probs(
        theta_score   = samp['theta_score'].values,
        theta_concede = samp['theta_concede'].values,
        t_remaining   = samp['t_remaining'].values,
        goal_diff     = samp['goal_difference'].values + shift,
    )
    err = max(np.abs(ref_w - samp[col_w].values).max(),
              np.abs(ref_d - samp[col_d].values).max())
    print(f'  shift {shift:+d}: max |one-pass − reference| = {err:.2e}')
    assert err < 1e-12, f'shift {shift:+d}: one-pass convolution disagrees with the reference'

print(f'One-pass convolution reproduces the reference at g, g+1 and g-1 ({len(samp):,} rows).')


In [ ]:
# ── Leverage Index formula ────────────────────────────────────────────────────
df['leverage_index_raw'] = (
    (df['p_win_plus1']  - df['p_win']).abs() +
    (df['p_draw_plus1'] - df['p_draw']).abs()
)

print('Raw Leverage Index statistics (all actions, pre-normalisation):')
print(df['leverage_index_raw'].describe().round(5))

# ── Quick sense-check by score state (all minutes) ────────────────────────────
# Level games should already have the highest mean LI even before normalisation.
print('\nMean raw LI by score state (all actions):')
print(df.groupby('score_state')['leverage_index_raw'].mean().round(5))


## 8. Normalise the One-Sided Index

The baseline index is rescaled so that its mean over actions from the 75th minute onward equals 1.0,
so `LI = 2` means twice as much at stake (on the scoring side) as an average late moment.

In [ ]:
# ── Identify post-75' actions (the clutch window) ─────────────────────────────
clutch_mask = df['is_post_75'] == 1
n_clutch    = clutch_mask.sum()
print(f"Post-75' actions: {n_clutch:,}  ({n_clutch / len(df) * 100:.1f}% of all actions)")

# ── Compute normalisation denominator ────────────────────────────────────────
# This is the mean raw LI across all clutch-window actions.
# After dividing, the post-75' mean will be exactly 1.0.
li_mean_post75 = df.loc[clutch_mask, 'leverage_index_raw'].mean()
print(f"Mean raw LI (post-75' actions, normalisation denominator): {li_mean_post75:.6f}")

# ── Apply normalisation to all rows ───────────────────────────────────────────
df['leverage_index'] = df['leverage_index_raw'] / li_mean_post75

# ── Verification ──────────────────────────────────────────────────────────────
post75_check = df.loc[clutch_mask, 'leverage_index'].mean()
print(f"\nPost-75' normalised LI mean: {post75_check:.6f}  (must be exactly 1.000000)")

print("\nFull-game normalised LI statistics:")
print(df['leverage_index'].describe().round(4))

print("\nPost-75' normalised LI statistics:")
print(df.loc[clutch_mask, 'leverage_index'].describe().round(4))


## 8b. Signed Expected-Points Leverage — the weights used by Stage 5

League points turn the outcome distribution into one number, the **expected points** from the current state:

```
W(g, n) = 3·P(win | g, n) + 1·P(draw | g, n)          0 ≤ W ≤ 3
```

A goal can be scored **or** conceded, and the two are not interchangeable. At 1–0 in the 89th minute a second goal is nearly worthless, while conceding is close to fatal. So we keep two separate weights:

```
L⁺(g, n) = W(g+1, n) − W(g, n)      gain if the acting team scores next
L⁻(g, n) = W(g, n) − W(g−1, n)      loss if the acting team concedes next
```

Both are in **league points** and are never negative. Stage 5 applies them channel by channel:

```
V(aᵢ) = ΔP_scores(aᵢ) · L⁺  −  ΔP_concedes(aᵢ) · L⁻
```

**Closed form, used as a check (S2).** Under the binomial model the differences reduce to draw probabilities:

```
L⁺(g) = 2·P(draw | g)   + P(draw | g+1)
L⁻(g) = 2·P(draw | g−1) + P(draw | g)
```

A goal that turns a draw into a win is worth 2 points (1 → 3); one that turns a loss into a draw is worth 1 (0 → 1). So even a level game is asymmetric: late on, `L⁺ ≈ 2·P(draw)` but `L⁻ ≈ P(draw)`.

**Expected behaviour at 85'+:**

| State | L⁺ | L⁻ |
|---|---|---|
| Level, g = 0 | ≈ 2·P(draw), high | ≈ P(draw), high |
| Trailing by one, g = −1 | high (equaliser) | low |
| Leading by one, g = +1 | low | high (as high as L⁺ when level) |
| Decided, \|g\| ≥ 3 | ≈ 0 | ≈ 0 |

The weights are **not normalised**: they stay in league points, so every action value downstream is in league points too.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SIGNED EXPECTED-POINTS LEVERAGE — the weights passed to Stage 5
# ══════════════════════════════════════════════════════════════════════════════
# W = 3·P(win) + 1·P(draw): expected league points from the current state
df['W']   = 3 * df['p_win'] + df['p_draw']
W_plus1   = 3 * df['p_win_plus1']  + df['p_draw_plus1']
W_minus1  = 3 * df['p_win_minus1'] + df['p_draw_minus1']

df['L_plus']  = W_plus1 - df['W']      # gain if the acting team scores next
df['L_minus'] = df['W'] - W_minus1     # loss if the acting team concedes next

late   = df['minute'] >= 85
lev_gd = (df[late & df['goal_difference'].between(-3, 3)]
          .groupby('goal_difference')[['L_plus', 'L_minus']].mean())

checks = []

# S1 — never negative: more goal difference cannot cost points
mn = df[['L_plus', 'L_minus']].min().min()
checks.append(('S1  L+ and L- are never negative', mn >= -1e-12, f'min = {mn:.2e}'))

# S2 — exact closed form of the binomial model
err = max((df['L_plus']  - (2 * df['p_draw'] + df['p_draw_plus1'])).abs().max(),
          (df['L_minus'] - (2 * df['p_draw_minus1'] + df['p_draw'])).abs().max())
checks.append(('S2  L+/L- match the closed form', err < 1e-9, f'max err = {err:.2e}'))

# S3 — one goal can move expected points by at most 3
mx = df[['L_plus', 'L_minus']].max().max()
checks.append(('S3  L+ and L- are at most 3 points', mx <= 3 + 1e-9, f'max = {mx:.3f}'))

# S4 — the asymmetry that motivates signing the weight
lead, trail = lev_gd.loc[1], lev_gd.loc[-1]
s4 = (lead['L_minus'] > lead['L_plus']) and (trail['L_plus'] > trail['L_minus'])
checks.append(("S4  85'+: leading L- > L+, trailing L+ > L-", s4,
               f"g=+1 {lead['L_plus']:.3f}/{lead['L_minus']:.3f}  "
               f"g=-1 {trail['L_plus']:.3f}/{trail['L_minus']:.3f}"))

# S5 — decided games carry almost no leverage
dead = df.loc[late & (df['goal_difference'].abs() >= 3), ['L_plus', 'L_minus']].mean().max()
checks.append(("S5  85'+ and |g| >= 3: mean L+/L- < 0.05", dead < 0.05, f'max mean = {dead:.4f}'))

print('=' * 84)
print('SIGNED LEVERAGE VALIDATION')
print('=' * 84)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<46} {detail}")
print('=' * 84)
assert all(ok for _, ok, _ in checks), 'Signed leverage checks failed — stop here.'

print("\nMean signed leverage at 85'+ by goal difference (league points):")
print(lev_gd.round(3).to_string())

lev_summary = (df.assign(window=np.where(df['is_post_75'] == 1, "post-75'", "pre-75'"))
                 .groupby(['window', 'score_state'])[['L_plus', 'L_minus']]
                 .mean().round(4))
print('\nMean signed leverage by window and score state:')
print(lev_summary.to_string())
lev_summary.to_csv(STAGE4_DIR / 'signed_leverage_by_score_state.csv')
print(f"\nSaved: {STAGE4_DIR / 'signed_leverage_by_score_state.csv'}")

In [ ]:
# ── Signed leverage over the match, and by scoreline at 85'+ ──────────────────
# Two series throughout, each always the same colour: L+ (scoring next) and
# L- (conceding next). Palette validated for colour-vision deficiency.
C_PLUS, C_MINUS = '#2a78d6', '#eb6834'
INK, INK_2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8), facecolor=SURFACE,
                         gridspec_kw={'width_ratios': [1, 1, 1, 1.25]})
panels = [(0, 'Level (g = 0)'), (1, 'Leading by one (g = +1)'), (-1, 'Trailing by one (g = −1)')]

for ax, (g_val, title) in zip(axes[:3], panels):
    by_min = (df[df['goal_difference'] == g_val]
              .groupby('minute')[['L_plus', 'L_minus']].mean())
    by_min = by_min[by_min.index <= 95]
    for col, colour, label in [('L_plus', C_PLUS, 'L+'), ('L_minus', C_MINUS, 'L−')]:
        ax.plot(by_min.index, by_min[col], color=colour, lw=2,
                solid_capstyle='round', solid_joinstyle='round', label=label)
        ax.annotate(label, (by_min.index[-1], by_min[col].iloc[-1]),
                    xytext=(6, 0), textcoords='offset points', va='center',
                    fontsize=10, color=INK)
    ax.set_title(title, fontsize=11, color=INK, loc='left')
    ax.set_xlabel('Match minute', color=INK_2)
    ax.set_xlim(0, 101)

axes[0].set_ylabel('League points', color=INK_2)
ymax = max(ax.get_ylim()[1] for ax in axes[:3])
for ax in axes[:3]:
    ax.set_ylim(0, ymax)

ax = axes[3]
x, w = np.arange(len(lev_gd)), 0.36
ax.bar(x - w / 2 - 0.01, lev_gd['L_plus'],  w, color=C_PLUS,  label='L+  scoring next')
ax.bar(x + w / 2 + 0.01, lev_gd['L_minus'], w, color=C_MINUS, label='L−  conceding next')
ax.set_xticks(x)
ax.set_xticklabels(['0' if g == 0 else f'{int(g):+d}' for g in lev_gd.index])
ax.set_xlabel('Goal difference, acting team', color=INK_2)
ax.set_title("At 85'+ by scoreline", fontsize=11, color=INK, loc='left')
ax.legend(frameon=False, fontsize=10, labelcolor=INK)

for ax in axes:
    ax.set_facecolor(SURFACE)
    ax.grid(axis='y', color=GRID, lw=1, ls='-')
    ax.grid(axis='x', visible=False)
    ax.set_axisbelow(True)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.tick_params(colors=INK_2, length=0)

fig.suptitle('Signed expected-points leverage: what the next goal is worth', x=0.01, ha='left',
             fontsize=13, color=INK)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'signed_leverage.png', dpi=150, bbox_inches='tight', facecolor=SURFACE)
plt.show()
print(f"Saved: {PLOTS_DIR / 'signed_leverage.png'}")

## 9. Save

`leverage_index.parquet` holds, for every action, the outcome probabilities, expected points `W`, the stakes
`L_plus` / `L_minus` (used by Stage 5), and the one-sided baseline `leverage_index`.

In [ ]:
OUTPUT_COLS = [
    # ── Match and player identity ──────────────────────────────────────────────
    'game_id', 'league_country', 'gameweek',
    'absolute_second', 'minute', 't_frame',
    'team_id', 'player_id',
    'is_home_team', 'is_post_75',
    # ── Game state at the moment of each action ────────────────────────────────
    'score_margin', 'score_state', 'goal_difference',
    # ── Logistic regression outputs (match-level P(score >=1)) ────────────────
    # Retained for traceability and debugging — not used directly in Stage 5.
    'p_will_score', 'p_will_concede',
    # ── Per-frame scoring rates ────────────────────────────────────────────────
    'theta_score', 'theta_concede',
    # ── Binomial-convolved win / draw / loss probabilities ────────────────────
    'p_win', 'p_draw', 'p_loss',
    # ── Counterfactual probabilities (goal_difference ± 1) ────────────────────
    'p_win_plus1', 'p_draw_plus1',
    'p_win_minus1', 'p_draw_minus1',
    # ── Signed expected-points leverage — the weights Stage 5 uses ───────────
    'W',                     # expected league points from the current state
    'L_plus',                # gain if the acting team scores next
    'L_minus',               # loss if the acting team concedes next
    # ── One-sided index — baseline for comparison ────────────────────────────
    'leverage_index_raw',    # before normalisation (retained for traceability)
    'leverage_index',        # normalised, post-75' mean = 1.0
]

out_path = LI_PATH
df[OUTPUT_COLS].to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {df[OUTPUT_COLS].shape}')

# ── Final verification checks ──────────────────────────────────────────────────
post75_li_final = df.loc[clutch_mask, 'leverage_index'].mean()
print(f"\nPost-75' normalised LI mean: {post75_li_final:.6f}  (must be 1.000000)")
print(f"Post-75' action count:        {clutch_mask.sum():,}")
print(f"Theta_score range:            {df['theta_score'].min():.5f} "
      f"to {df['theta_score'].max():.5f}")
print(f"LI range (post-75'):          {df.loc[clutch_mask, 'leverage_index'].min():.4f} "
      f"to {df.loc[clutch_mask, 'leverage_index'].max():.4f}")
print(f"L+ range (league points):     {df['L_plus'].min():.4f} to {df['L_plus'].max():.4f}")
print(f"L- range (league points):     {df['L_minus'].min():.4f} to {df['L_minus'].max():.4f}")

## 10. Outputs

| File | Description |
|---|---|
| `data/leverage_index.parquet` | Per-action outcome probabilities, `W`, stakes `L_plus` / `L_minus`, and the one-sided baseline. Input to Stage 5. |
| `stage4_winprob/winprob_fold_metrics.csv` | Out-of-league Brier score and ROC-AUC per fold. |
| `stage4_winprob/signed_leverage_by_score_state.csv` | Mean `L_plus` / `L_minus` by window and score state. |
| `stage4_winprob/plots/win_prob_calibration.png` | Calibration and `P(win)` by score state. |
| `stage4_winprob/plots/signed_leverage.png` | Stakes over the match and by scoreline at 85'+. |

**Next → Stage 5:** joins these stakes with the Stage 3 action values.